In [ ]:
import os
import trafilatura
from pathlib import Path
from config import Arch_SOURCES # URLs of sources

def download_sources(SOURCE, base_dir):
    for category, urls in SOURCE.items():
        base_dir.mkdir(parents=True, exist_ok=True)
        category_path = base_dir / category
        category_path.mkdir(parents=True, exist_ok=True)
        
        print(f"--- Working category: {category.upper()} ---")
        
        for url in urls:
            file_name = url.split("/")[-1] + ".txt"
            target_file = category_path / file_name
            
            print(f"load: {url}...")
            
            try:
                downloaded = trafilatura.fetch_url(url)
                content = trafilatura.extract(downloaded, include_tables=True, include_comments=False)
                
                if content:
                    with open(target_file, "w", encoding="utf-8") as f:
                        f.write(content)
                    print(f"Succesfully saved: {target_file}")
                else:
                    print(f"Warning: no content was found for {url}.")
                    
            except Exception as e:
                print(f"Error during loading {url}: {e}")


# merges all sources to one .txt

def merge_txt_files(source_dir, output_file):
    base_path = Path(source_dir).resolve()
    target_path = Path(output_file).resolve()
    files = [f for f in base_path.rglob("*.txt")]
    
    print(f"found data: {len(files)}")
    
    if not files:
        print("No data found!")
        return

    with open(target_path, "w", encoding="utf-8") as outfile:
        for file_path in files:
            display_name = file_path.relative_to(base_path)
            
            outfile.write(f"\n\n=== TOPIC: {display_name} ===\n")
            outfile.write(f"path: {file_path}\n")
            outfile.write("-" * 30 + "\n\n")
            
            try:
                with open(file_path, "r", encoding="utf-8") as infile:
                    content = infile.read()
                    outfile.write(content)
                print(f"add: {display_name}")
            except Exception as e:
                print(f"can not read {display_name}: {e}")


--- Verarbeite Kategorie: CORE ---
Lade: https://wiki.archlinux.org/title/Installation_guide...
Erfolgreich gespeichert: /workspaces/Arch_PC_Assistent/sources/raw_Arch_source/core/Installation_guide.txt
Lade: https://wiki.archlinux.org/title/General_recommendations...
Erfolgreich gespeichert: /workspaces/Arch_PC_Assistent/sources/raw_Arch_source/core/General_recommendations.txt
Lade: https://wiki.archlinux.org/title/Users_and_groups...
Erfolgreich gespeichert: /workspaces/Arch_PC_Assistent/sources/raw_Arch_source/core/Users_and_groups.txt
Lade: https://wiki.archlinux.org/title/Pacman...
Erfolgreich gespeichert: /workspaces/Arch_PC_Assistent/sources/raw_Arch_source/core/Pacman.txt
Lade: https://wiki.archlinux.org/title/Arch_User_Repository...
Erfolgreich gespeichert: /workspaces/Arch_PC_Assistent/sources/raw_Arch_source/core/Arch_User_Repository.txt
Lade: https://wiki.archlinux.org/title/Systemd...
Erfolgreich gespeichert: /workspaces/Arch_PC_Assistent/sources/raw_Arch_source/core/Syste

In [ ]:
# download sources from given URL
base_dir = Path("/workspaces/Arch_PC_Assistent/sources/raw_Arch_source")
download_sources(Arch_SOURCES, base_dir)

# concatenate sources into on .txt
source_folder = Path("/workspaces/Arch_PC_Assistent/sources/raw_Arch_source" )
output_path = Path("/workspaces/Arch_PC_Assistent/sources/Arch_Wiki_Combined.txt") 

if not os.path.exists(source_folder):
    print(f"Folder '{source_folder}' does not existing")
else:
    merge_txt_files(source_folder, output_path)
    print(f"Done! Check: {output_path}")

found data: 45
add: core/Installation_guide.txt
add: core/General_recommendations.txt
add: core/Users_and_groups.txt
add: core/Pacman.txt
add: core/Arch_User_Repository.txt
add: core/Systemd.txt
add: core/System_maintenance.txt
add: core/General_troubleshooting.txt
add: core/AUR_helpers.txt
add: core/Tips_and_tricks.txt
add: core/Kernel_parameters.txt
add: desktop/Wayland.txt
add: desktop/NetworkManager.txt
add: desktop/PipeWire.txt
add: desktop/Environment_variables.txt
add: desktop/XDG_Desktop_Portal.txt
add: desktop/Fonts.txt
add: desktop/Hyprland.txt
add: desktop/SDDM.txt
add: desktop/Desktop_notifications.txt
add: desktop/Cursor_themes.txt
add: filesystem/Btrfs.txt
add: filesystem/Partitioning.txt
add: filesystem/Fstab.txt
add: filesystem/Swap.txt
add: filesystem/Ext4.txt
add: hardware/NVIDIA.txt
add: hardware/AMDGPU.txt
add: hardware/Intel_graphics.txt
add: hardware/Bluetooth.txt
add: hardware/Microcode.txt
add: hardware/Laptop.txt
add: hardware/Power_management.txt
add: bootload

In [ ]:
import os
import json
import time
import re
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

# 1. Setup
load_dotenv()
client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1"
)

def get_smart_chunks(full_text, max_chars=3500):
    topics = full_text.split("=== THEMA:")
    final_chunks = []
    for topic_content in topics:
        if not topic_content.strip(): continue
        full_topic = "=== THEMA:" + topic_content
        if len(full_topic) <= max_chars:
            final_chunks.append(full_topic)
        else:
            current_chunk = ""
            paragraphs = full_topic.split("\n\n")
            for para in paragraphs:
                if len(current_chunk) + len(para) < max_chars:
                    current_chunk += para + "\n\n"
                else:
                    final_chunks.append(current_chunk.strip())
                    current_chunk = para + "\n\n"
            if current_chunk: final_chunks.append(current_chunk.strip())
    return final_chunks

def generate_data(text_chunk, retries=3):
    system_prompt = """You are a Senior AI Training Engineer.  

        Your mission is to produce a GOLD STANDARD SFT (Supervised 
        Fine-Tuning) dataset for a high-performance Arch Linux AI Assistant. 

        STRICT DATA STRUCTURE: 
        Return a JSON object with a "tasks" key containing an array of 5 objects. 
        Each object MUST contain exactly these three keys: 
        1. "instruction": A clear, diverse, and technically specific Arch Linux task or question. 

        2. "thought": A deep expert-level 'Chain of Thought'. Explain 
            the technical background, weigh alternatives, and show the logical steps
            to the solution. 
        3. "output": The final professional answer, formatted in Markdown with code blocks for commands. 

        CRITICAL JSON RULES (To prevent parsing errors): 
        - ESCAPE all double quotes inside strings using \\" (e.g., "echo \\"hello\\""). 
        - Use \\n for all newlines. NEVER use literal line breaks inside a JSON string. 
        - Ensure every task is a complete, valid JSON object. 

        TECHNICAL FOCUS: 
        - Use the provided CONTEXT as the anchor. 
        - Supplement with expert knowledge for completeness. 
        - Follow the 'Arch Way' (correctness, simplicity, and modern tools)."""

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model="deepseek-v4-pro",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": f"CONTEXT:\n{text_chunk}"}
                ],
                response_format={ "type": "json_object" }, # force to return a json-obj but without guarantee
                temperature=0.2,
                max_tokens=7000
            )
            
            raw_content = response.choices[0].message.content
            if not raw_content:
                continue
                
            # JSON bereinigen
            clean_content = re.sub(r'^```json\s*|\s*```$', '', raw_content.strip(), flags=re.MULTILINE)
            
            try:
                data = json.loads(clean_content)
                return data.get("tasks", []), raw_content, True
            except json.JSONDecodeError:
                match = re.search(r'(\{.*\})', clean_content, re.DOTALL)
                if match:
                    data = json.loads(match.group(1))
                    return data.get("tasks", []), raw_content, True
                raise

        except Exception as e:
            if attempt == retries - 1: # Letzter Versuch fehlgeschlagen
                return [], str(raw_content if 'raw_content' in locals() else "Error"), False
            time.sleep(5 * (attempt + 1))
            
    return [], "Timeout/Error", False

def safe_save_json(data, path):
    """Speichert die JSON-Datei atomar."""
    temp_path = path.with_suffix(".tmp")
    with open(temp_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    os.replace(temp_path, path)

def log_raw_output(raw_text, success, chunk_idx):
    """Speichert jede API-Antwort in einer Textdatei mit Erfolgsstatus."""
    log_path = Path("/workspaces/Arch_PC_Assistent/dataset/raw_responses.txt")
    status = "ERFOLGREICH IN JSON" if success else "FEHLER (NUR TEXT)"
    
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(f"\n{'='*60}\n")
        f.write(f"CHUNK: {chunk_idx} | STATUS: {status} | ZEIT: {time.ctime()}\n")
        f.write(f"{'='*60}\n")
        f.write(raw_text)
        f.write("\n")

def main():
    base_dir = Path("/workspaces/Arch_PC_Assistent")
    source_file = base_dir / "sources/Arch_Wiki_Combined.txt"
    output_path = base_dir / "dataset/arch_dataset_progress.json"
    checkpoint_path = base_dir / "dataset/last_chunk.txt"
    output_path.parent.mkdir(parents=True, exist_ok=True)

    all_data = []
    if output_path.exists():
        with open(output_path, "r", encoding="utf-8") as f:
            try:
                all_data = json.load(f)
                print(f"Resuming: Loaded {len(all_data)} entries.")
            except:
                print("Starte neuen Datensatz.")

    start_index = 0
    if checkpoint_path.exists():
        with open(checkpoint_path, "r") as f:
            try: start_index = int(f.read().strip())
            except: start_index = 0

    with open(source_file, "r", encoding="utf-8") as f:
        chunks = get_smart_chunks(f.read())

    print(f"Arch Factory: {len(chunks)} Chunks total. Start bei {start_index}")
    print("-" * 50)

    for i in range(start_index, len(chunks)):
        chunk = chunks[i]
        title = (chunk.splitlines()[0][:30] + "..") if chunk.splitlines() else "Unbekannt"
        
        # API-Abfrage
        tasks, raw_text, success = generate_data(chunk)
        
        # 1. IMMER die Rohdaten loggen (egal ob Erfolg oder nicht)
        log_raw_output(raw_text, success, i + 1)
        
        if success and tasks:
            # 2. In die Haupt-Liste aufnehmen
            all_data.extend(tasks)
            # 3. Atomares Speichern der JSON
            safe_save_json(all_data, output_path)
            status_msg = f"+{len(tasks)} Tasks"
        else:
            status_msg = "FEHLER (Rohdaten gespeichert)"

        # Checkpoint aktualisieren
        with open(checkpoint_path, "w") as f:
            f.write(str(i + 1))
        
        progress = (i + 1) / len(chunks)
        bar = "#" * int(progress * 20) + "-" * (20 - int(progress * 20))
        print(f"[{bar}] {progress:>4.1%} | {title:<32} | {status_msg}")
        
        time.sleep(1)

    print("-" * 60)
    print(f"FERTIG! Datensaetze insgesamt: {len(all_data)}")

if __name__ == "__main__":
    main()

Resuming: Loaded 670 entries.
Starting Arch AI Factory: 134 chunks total.
--------------------------------------------------
🚀 Arch AI Factory gestartet | Ziel: 134 Chunks
------------------------------------------------------------
[███████████████░░░░░] 79.1% | === THEMA: bootloader/Unified_Exten.. | +5 | $0.0252
------------------------------------------------------------
✅ FERTIG! Gesamtkosten: $0.03 | Datensätze: 675


In [44]:
import json
from pathlib import Path

file_path = Path("/workspaces/Arch_PC_Assistent/dataset/arch_troubleshooting_dataset.json")

def validate():
    if not file_path.exists():
        print("❌ Datei nicht gefunden!")
        return

    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        
        print(f"✅ JSON-Syntax ist korrekt. {len(data)} Samples gefunden.")
        
        errors = 0
        for i, item in enumerate(data):
            missing = [key for key in ["instruction", "thought", "output"] if key not in item]
            if missing:
                print(f"⚠️ Sample {i} fehlerhaft! Fehlende Keys: {missing}")
                errors += 1
        
        if errors == 0:
            print("🚀 Alle Samples sind strukturell perfekt für das Training!")
        else:
            print(f"❌ Insgesamt {errors} fehlerhafte Samples gefunden.")

    except json.JSONDecodeError as e:
        print(f"❌ Schwerer JSON-Syntaxfehler: {e}")

if __name__ == "__main__":
    validate()

✅ JSON-Syntax ist korrekt. 402 Samples gefunden.
🚀 Alle Samples sind strukturell perfekt für das Training!


In [41]:
import json
from pathlib import Path

# Pfade anpassen
input_path = Path("/workspaces/Arch_PC_Assistent/dataset/arch_dataset_progress.json")
output_path = Path("/workspaces/Arch_PC_Assistent/dataset/arch_dataset_cleaned.json")

def fix():
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    # Behalte nur Samples, die alle 3 Keys haben
    required_keys = {"instruction", "thought", "output"}
    cleaned_data = [item for item in data if all(k in item for k in required_keys)]
    
    removed = len(data) - len(cleaned_data)
    
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(cleaned_data, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Bereinigung abgeschlossen.")
    print(f"   Ursprünglich: {len(data)} Samples")
    print(f"   Entfernt: {removed} fehlerhafte Samples")
    print(f"   Final: {len(cleaned_data)} Samples gespeichert in {output_path.name}")

if __name__ == "__main__":
    fix()

✅ Bereinigung abgeschlossen.
   Ursprünglich: 675 Samples
   Entfernt: 1 fehlerhafte Samples
   Final: 674 Samples gespeichert in arch_dataset_cleaned.json


In [43]:
import os
import json
import time
import re
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

# 1. Setup
load_dotenv()
client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1"
)

def get_smart_chunks(full_text, max_chars=3500):
    topics = full_text.split("=== THEMA:")
    final_chunks = []
    for topic_content in topics:
        if not topic_content.strip(): continue
        full_topic = "=== THEMA:" + topic_content
        if len(full_topic) <= max_chars:
            final_chunks.append(full_topic)
        else:
            current_chunk = ""
            paragraphs = full_topic.split("\n\n")
            for para in paragraphs:
                if len(current_chunk) + len(para) < max_chars:
                    current_chunk += para + "\n\n"
                else:
                    final_chunks.append(current_chunk.strip())
                    current_chunk = para + "\n\n"
            if current_chunk: final_chunks.append(current_chunk.strip())
    return final_chunks

def generate_troubleshooting_data(text_chunk, retries=3):
    """Spezieller Prompt für Fehlerszenarien und Reparaturen."""
    system_prompt = """You are a Senior Arch Linux Support Engineer (Level 3).
    Your mission is to produce a TROUBLESHOOTING SFT dataset for expert recovery.

    TASK:
    Based on the CONTEXT, create 3 unique scenarios where something fails (errors, boot loops, driver crashes, misconfigurations).

    STRUCTURE:
    Return a JSON object with a "tasks" key. Each task MUST have:
    1. "instruction": A realistic user report or error message (e.g., 'My screen is black after NVIDIA update' or 'UUID mismatch on boot').
    2. "thought": A deep diagnostic analysis. Identify the root cause based on the context, mention relevant logs (journalctl, dmesg), and plan the recovery.
    3. "output": A professional, step-by-step recovery guide using terminal commands and Arch best practices.

    CRITICAL RULES:
    - Focus on 'What can go wrong' in this specific context.
    - ESCAPE all double quotes with \\". 
    - Use \\n for newlines. 
    - Respond ONLY with the JSON object."""

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model="deepseek-v4-pro",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": f"CONTEXT:\n{text_chunk}"}
                ],
                response_format={ "type": "json_object" },
                temperature=0.3,
                max_tokens=5000 
            )
            
            raw_content = response.choices[0].message.content
            clean_content = re.sub(r'^```json\s*|\s*```$', '', raw_content.strip(), flags=re.MULTILINE)
            
            data = json.loads(clean_content)
            return data.get("tasks", [])
            
        except Exception:
            time.sleep(5 * (attempt + 1))
            
    return []

def main():
    base_dir = Path("/workspaces/Arch_PC_Assistent")
    source_file = base_dir / "sources/Arch_Wiki_Combined.txt"
    output_path = base_dir / "dataset/arch_troubleshooting_dataset.json"
    checkpoint_path = base_dir / "dataset/last_chunk_troubleshoot.txt"
    output_path.parent.mkdir(parents=True, exist_ok=True)

    all_data = []
    seen_instructions = set()
    if output_path.exists():
        with open(output_path, "r", encoding="utf-8") as f:
            try:
                all_data = json.load(f)
                for item in all_data:
                    if "instruction" in item:
                        seen_instructions.add(item["instruction"].strip().lower())
                print(f"Phase 2 Resume: Geladen {len(all_data)} Troubleshooting-Eintraege.")
            except: pass

    start_index = 0
    if checkpoint_path.exists():
        with open(checkpoint_path, "r") as f:
            try: 
                start_index = int(f.read().strip())
            except: 
                start_index = 0

    if not source_file.exists():
        print(f"Fehler: Quelldatei {source_file} nicht gefunden.")
        return

    with open(source_file, "r", encoding="utf-8") as f:
        chunks = get_smart_chunks(f.read())

    print(f"Arch Troubleshooting Factory - Start bei Chunk {start_index}/{len(chunks)}")
    print("-" * 60)

    for i in range(start_index, len(chunks)):
        chunk = chunks[i]
        title_line = chunk.splitlines()[0] if chunk.splitlines() else "Unbekannt"
        title = (title_line[:35] + '..') if len(title_line) > 35 else title_line
        
        start_time = time.time()
        results = generate_troubleshooting_data(chunk)
        
        if results:
            new_entries = 0
            for task in results:
                if isinstance(task, dict) and "instruction" in task:
                    instr_clean = str(task["instruction"]).strip().lower()
                    if instr_clean not in seen_instructions:
                        all_data.append(task)
                        seen_instructions.add(instr_clean)
                        new_entries += 1
            
            with open(output_path, "w", encoding="utf-8") as f:
                json.dump(all_data, f, indent=2, ensure_ascii=False)
            with open(checkpoint_path, "w") as f:
                f.write(str(i + 1))
            
            progress = (i + 1) / len(chunks)
            print(f"[{progress:>4.1%}] {title:<37} | +{new_entries} Fixes")
        else:
            print(f"[{i+1}/{len(chunks)}] Warnung: Fehler bei Chunk '{title}'")
        
        time.sleep(1)

    print("-" * 60)
    print(f"Phase 2 abgeschlossen! Troubleshooting-Eintraege insgesamt: {len(all_data)}")

if __name__ == "__main__":
    main()

Arch Troubleshooting Factory - Start bei Chunk 0/134
------------------------------------------------------------
[0.7%] === THEMA: core/Installation_guide... | +3 Fixes
[1.5%] Installation guide                    | +3 Fixes
[2.2%] Unbekannt                             | +3 Fixes
[3.0%] === THEMA: core/General_recommendat.. | +3 Fixes
[3.7%] General recommendations               | +3 Fixes
[4.5%] Unbekannt                             | +3 Fixes
[5.2%] === THEMA: core/Users_and_groups.tx.. | +3 Fixes
[6.0%] Users and groups                      | +3 Fixes
[6.7%] Unbekannt                             | +3 Fixes
[7.5%] === THEMA: core/Pacman.txt ===        | +3 Fixes
[8.2%] pacman                                | +3 Fixes
[9.0%] Unbekannt                             | +3 Fixes
[9.7%] === THEMA: core/Arch_User_Repositor.. | +3 Fixes
[10.4%] Arch User Repository                  | +3 Fixes
[11.2%] Unbekannt                             | +3 Fixes
[11.9%] === THEMA: core/Systemd.txt ===     